In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install requests pandas pyarrow pycountry -q
# import gbif_occurrences

In [ ]:
#!/usr/bin/env python3
"""
gbif_occurrences.py  —  Google Colab
====================================
Tier-3 RISK layer for Paper 1. Pulls OBSERVED tick distributions from GBIF per
species, aggregates to country (and optional 1-degree grid), so the corpus can
be joined against "where ticks actually are" — the empirical half of the
research-vs-risk misalignment claim.

INPUT species list: your extracted tick species (ent_ticks / triples). GBIF is
queried per species via its match + occurrence-facet API. Country counts come
from GBIF's facet aggregation (one light call per species), so this is fast and
does NOT download millions of raw records.

OUTPUT
  1. gbif_species_country.parquet   species x country -> occurrence_count
  2. gbif_country_richness.parquet  country -> n_tick_species, total_occurrences
  3. merged onto corpus: for each paper's study country, the observed tick
     richness / occurrence intensity there (a Paper-1 risk covariate)

APPROACH: GBIF REST API, keyless. species/match to get a taxonKey, then
occurrence/search facet=country with occurrenceStatus=PRESENT. Resumable cache.

SETUP
  !pip install requests pandas pyarrow pycountry -q
"""

import json, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd, requests, pycountry

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_methods_triples.parquet"
OUT_SPECIES  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/GBIF/gbif_species_country.parquet"
OUT_RICHNESS = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/GBIF/gbif_country_richness.parquet"
OUT_MERGED   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_risk.parquet"
GBIF_CACHE   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/GBIF/gbif_cache.jsonl"
WORKERS      = 6

Path(GBIF_CACHE).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_SPECIES).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_RICHNESS).parent.mkdir(parents=True, exist_ok=True)

# tick genera to keep (so we query real tick taxa, not stray matches)
TICK_GENERA = {"ixodes","amblyomma","rhipicephalus","hyalomma","dermacentor",
               "haemaphysalis","boophilus","argas","ornithodoros","otobius",
               "nuttalliella","bothriocroton","carios","antricola","margaropus",
               "rhipicentor","nosomma","anomalohimalaya","cosmiomma"}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)


# =========================================================================== #
# 1. build the species list from the corpus (triples / ent_ticks)             #
# =========================================================================== #
def species_list(df):
    sp = set()
    cols = [c for c in ("ent_ticks", "triples_strong") if c in df.columns]
    for c in cols:
        for v in df[c].dropna():
            for token in S(v).replace("||", ";").replace("|", ";").split(";"):
                token = token.strip().lower()
                # keep binomials (Genus species) whose genus is a tick genus
                parts = token.split()
                if len(parts) >= 2 and parts[0] in TICK_GENERA:
                    sp.add(f"{parts[0]} {parts[1]}")
                elif token in TICK_GENERA:
                    sp.add(token)  # genus-level fallback
    print(f"[gbif] {len(sp):,} distinct tick taxa to query", file=sys.stderr)
    return sorted(sp)


# =========================================================================== #
# 2. GBIF: match name -> taxonKey, then country facet counts                   #
# =========================================================================== #
def load_cache():
    done = {}
    if Path(GBIF_CACHE).exists():
        for line in open(GBIF_CACHE):
            try:
                d = json.loads(line); done[d["species"]] = d
            except Exception:
                pass
    print(f"[gbif] {len(done):,} species cached", file=sys.stderr)
    return done

def _match(name, sess):
    r = sess.get("https://api.gbif.org/v1/species/match",
                 params={"name": name, "strict": "false"}, timeout=30)
    if r.status_code == 200:
        j = r.json()
        if j.get("matchType") != "NONE" and j.get("usageKey"):
            return j["usageKey"], j.get("rank", ""), j.get("scientificName", "")
    return None, "", ""

def _country_facets(taxon_key, sess):
    # facet=country returns counts per ISO2, PRESENT records only
    r = sess.get("https://api.gbif.org/v1/occurrence/search",
                 params={"taxonKey": taxon_key, "occurrenceStatus": "PRESENT",
                         "facet": "country", "facetLimit": 300, "limit": 0},
                 timeout=45)
    out = {}
    if r.status_code == 200:
        for f in r.json().get("facets", []):
            if f.get("field") == "COUNTRY":
                for c in f.get("counts", []):
                    out[c["name"]] = c["count"]   # name is ISO2 here
    return out

def harvest(species, cache):
    todo = [s for s in species if s not in cache]
    print(f"[gbif] {len(todo):,} to fetch", file=sys.stderr)
    sess = requests.Session()
    fh = open(GBIF_CACHE, "a")

    def one(name):
        for attempt in range(3):
            try:
                key, rank, sci = _match(name, sess)
                if not key:
                    return {"species": name, "taxonKey": None, "countries": {}}
                fac = _country_facets(key, sess)
                return {"species": name, "taxonKey": key, "rank": rank,
                        "matched": sci, "countries": fac}
            except requests.RequestException:
                time.sleep(2 ** attempt)
        return {"species": name, "taxonKey": None, "countries": {}}

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(one, s) for s in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            d = fut.result()
            fh.write(json.dumps(d, ensure_ascii=False) + "\n"); fh.flush()
            cache[d["species"]] = d
            if n % 50 == 0:
                print(f"[gbif] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return cache


# =========================================================================== #
# 3. build species-country + country-richness tables                          #
# =========================================================================== #
def build_tables(cache):
    rows = []
    for sp, d in cache.items():
        for iso2, cnt in (d.get("countries") or {}).items():
            rows.append({"species": sp, "iso2": iso2, "occurrence_count": cnt})
    sc = pd.DataFrame(rows)
    sc.to_parquet(OUT_SPECIES, index=False)

    rich = (sc.groupby("iso2")
              .agg(n_tick_species=("species", "nunique"),
                   total_occurrences=("occurrence_count", "sum"))
              .reset_index())
    rich.to_parquet(OUT_RICHNESS, index=False)
    print(f"[gbif] {sc['species'].nunique()} species x {rich['iso2'].nunique()} "
          f"countries; richness table {len(rich):,} rows", file=sys.stderr)
    return sc, rich


# =========================================================================== #
# 4. attach observed tick risk to each paper's study country                  #
# =========================================================================== #
def attach(df, rich):
    r = rich.set_index("iso2")
    def row(rw):
        sc = [c for c in S(rw.get("study_countries")).split(";") if c]
        if not sc or sc[0] not in r.index:
            return pd.Series({"gbif_tick_richness": None, "gbif_tick_occurrences": None})
        return pd.Series({"gbif_tick_richness": int(r.loc[sc[0], "n_tick_species"]),
                          "gbif_tick_occurrences": int(r.loc[sc[0], "total_occurrences"])})
    out = pd.concat([df, df.apply(row, axis=1)], axis=1)
    out.to_parquet(OUT_MERGED, index=False)
    print(f"[write] {OUT_MERGED}", file=sys.stderr)
    print(f"[coverage] papers with observed tick richness: "
          f"{out['gbif_tick_richness'].notna().mean():.1%}", file=sys.stderr)
    return out


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    species = species_list(df)
    cache = harvest(species, load_cache())
    sc, rich = build_tables(cache)
    merge = attach(df, rich)

In [ ]:
species

In [ ]:
cache

In [ ]:
merge = attach(df, rich)
merge